In [7]:
!pip install --upgrade "kaggle-environments>=1.28.0"


In [8]:
from kaggle_environments import make

env = make("orbit_wars", debug=True) #make() initializes the environment and stuff
print(f"Environment: {env.name} v{env.version}")
print(f"Players: {env.specification.agents}")
print(f"Max steps: {env.configuration.episodeSteps}")

Environment: orbit_wars v1.0.9
Players: [2, 4]
Max steps: 500


## Understanding the Observation

Each turn your agent receives an observation with:
- `player` - your player ID (0-3)
- `planets` - list of `[id, owner, x, y, radius, ships, production]`
- `fleets` - list of `[id, owner, x, y, angle, from_planet_id, ships]`
- `angular_velocity` - how fast inner planets rotate (radians/turn)

Your agent returns a list of moves: `[from_planet_id, angle_in_radians, num_ships]`

In [9]:
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet
from typing import NamedTuple

class planet_information:
    def __init__(self, raw_list, maxSpeed):
        # Extract base metrics passed from your distance solvers
        self.best_planet = raw_list[0]
        self.best_target = raw_list[1]
        self.best_distance = raw_list[2]
        
        # Rotational vectors pack 6 indices containing future path tracking
        if len(raw_list) >= 6:
            self.best_angle = raw_list[3]
            self.x = raw_list[4]
            self.y = raw_list[5]
            # Handle if best_time was appended to the list, or calculate it
            self.best_time = raw_list[6] if len(raw_list) > 6 else (self.best_distance / maxSpeed)
        else:
            # Static nodes require direct coordinate and distance math logs
            self.best_angle = math.atan2((self.best_target.y - self.best_planet.y), 
                                         (self.best_target.x - self.best_planet.x))
            self.x = self.best_target.x
            self.y = self.best_target.y
            self.best_time = self.return_static_best_time(maxSpeed)
            
    def return_static_best_time(self, maxSpeed):
        # FIXED: Corrected destination target mapping pointers
        distance = math.dist((self.best_planet.x, self.best_planet.y), (self.best_target.x, self.best_target.y))
        # Exact Orbit Wars fleet velocity log scaling
        speed = 1.0 + (maxSpeed - 1.0) * (math.log(max(2, self.best_planet.ships)) / math.log(1000))**1.5
        return distance / max(0.1, speed)
        
def barbell_action_plan(obs, configuration):
    moves = []
    player = obs.player
    raw_planets = obs.planets

    planets = [Planet(*p) for p in raw_planets]

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    angular_velocity = obs.angular_velocity
    maxSpeed = configuration.shipSpeed
    turns = obs.step

    moves = planet_seeker(my_planets=my_planets, targets=targets, maxSpeed=maxSpeed, angular_velocity=angular_velocity, turns=turns)

    return moves

def planet_seeker(my_planets, targets, maxSpeed, angular_velocity, turns):
    targets_static = []
    targets_rotational = []
    moves = []
    for target in targets:
        r = math.dist((target.x, target.y), (50, 50))
        priority_score = max(1, target.production)
        
        if r < 50:
            # Symmetrical Layout: [Planet_Instance, radius, score]
            target_info = [target, r, priority_score]
            targets_rotational.append(target_info)
        else:
            # Symmetrical Layout: [Planet_Instance, radius, score]
            # Even for static planets, keeping 'r' maintains identical indexing
            target_info = [target, r, priority_score]
            targets_static.append(target_info)

    targets_static.sort(key=lambda t: t[2], reverse=True)
    targets_rotational.sort(key=lambda t: t[2], reverse=True)

    best_static = compute_distance_static(my_planets=my_planets, 
                                          targets_static=targets_static)
    best_rotational = compute_distance_rotational(my_planets=my_planets,
                                                 targets_rotational=targets_rotational,
                                                 angular_velocity=angular_velocity,
                                                 maxSpeed=maxSpeed,
                                                 turns=turns)
    best_static = planet_information(raw_list=best_static, maxSpeed=maxSpeed)
    best_rotational = planet_information(raw_list=best_rotational, maxSpeed=maxSpeed)

    if best_static:
        ships_needed_static = best_static.best_target.ships + (best_static.best_target.production * best_static.best_time) + 2
        safety_floor = max(30, best_static.best_planet.ships * 0.50)
        if best_static.best_planet.ships - ships_needed_static >= safety_floor:
            move_schema = move_compute(best_planet=best_static.best_planet,
                                 best_target=best_static.best_target, 
                                 ships_needed=ships_needed_static)
            moves.append(move_schema)
        else:
            pass

    if best_rotational:
        # FIXED: All parameter lines re-mapped to extract strictly from best_rotational
        ships_needed = best_rotational.best_target.ships + (best_rotational.best_target.production * best_rotational.best_time) + 2
        
        # FIXED: Lowered safety floor (25% / 15 ships) running cleanly on the correct home planet
        safety_floor = max(15, best_rotational.best_planet.ships * 0.25)
        
        if best_rotational.best_planet.ships - ships_needed >= safety_floor:
            ships_needed = 2.0 * ships_needed
            move_schema = move_compute(best_planet=best_rotational.best_planet,
                                      best_target=best_rotational.best_target,
                                      ships_needed=ships_needed)
            moves.append(move_schema)
        else:
            # FIXED: Converted to pass to prevent an artificial system freeze
            pass 

    return moves
            


def move_compute(best_planet, best_target, ships_needed):
    angle = math.atan2((best_planet.y - best_target.y), 
                       (best_planet.x - best_target.x))
    return [best_planet.id, angle, ships_needed]
        
# 1. Define a clean, high-performance data container interface
class StaticMatch(NamedTuple):
    best_planet: object
    best_target: object
    best_distance: float
    best_time: float # Added: Your upstream logic explicitly checks `.best_time`

def compute_distance_static(my_planets, targets_static, maxSpeed=1.0):
    best_planet = None
    best_distance = float('inf')
    best_target = None
    best_time = 0.0

    for planet in my_planets:
        # Calculate dynamic launch speed parameters per planet asset
        safe_ship_count = max(2, planet.ships)
        speed = 1.0 + (maxSpeed - 1.0) * (math.log(safe_ship_count) / math.log(1000))**1.5
        
        for target_info in targets_static:
            target = target_info[0]
            distance = math.dist((planet.x, planet.y), (target.x, target.y))
            if distance < best_distance:
                best_planet = planet
                best_target = target
                best_distance = distance
                best_time = distance / speed # Automatically calculate transit time metrics

    # Returns an object that perfectly matches your upstream dot-notation layout
    return StaticMatch(best_planet, best_target, best_distance, best_time)


def compute_distance_rotational(my_planets, 
                                targets_rotational, 
                                angular_velocity, 
                                maxSpeed, 
                                turns):
    best_planet = None
    best_distance = float('inf')
    best_target = None
    best_angle = None
    best_x = None
    best_y = None
    best_time = None

        # Fully audited and cleaned production pipeline
    for planet in my_planets:
        # Prevent domain crashes if ship inventory assets are depleted
        safe_ship_count = max(2, planet.ships)
        speed = 1.0 + (maxSpeed - 1.0) * (math.log(safe_ship_count) / math.log(1000))**1.5
        
        for target_info in targets_rotational:
            # FIX: Explicit variable decoupling to prevent array name collisions
            target = target_info[0]
            r = target_info[1]
            
            init_distance = math.dist((target.x, target.y), (planet.x, planet.y))
            init_angle = math.atan2(target.y - planet.y, target.x - planet.x)
            init_time = init_distance / speed

            # FIX: Total conversion to clean 4-space indentation structure
            est_angle = init_angle + (angular_velocity * init_time)
            est_x = r * math.cos(est_angle) + 50
            est_y = r * math.sin(est_angle) + 50
            estimated_distance = math.dist((est_x, est_y), (planet.x, planet.y))
            est_time = estimated_distance / speed

            # FIX: Added required math prefix namespaces to trig modules
            final_angle = est_angle + (angular_velocity * est_time)
            final_x = r * math.cos(final_angle) + 50
            final_y = r * math.sin(final_angle) + 50
            final_distance = math.dist((final_x, final_y), (planet.x, planet.y))
            final_time = est_time
            if final_distance < best_distance:
                best_planet = planet
                best_target = target
                best_distance = final_distance
                best_x = final_x
                best_y = final_y
                best_time = final_time


    return [best_planet, best_target, best_distance, best_angle, best_x, best_y, best_time]




In [10]:
from kaggle_environments import make
# Run a quick game to see what the observation looks like
env = make("orbit_wars", debug=True)
env.run(["random", "random"]) #so env runs live updates for every planet or something

# Peek at the initial observation

from kaggle_environments.envs.orbit_wars.orbit_wars import Planet

obs = env.steps[1][0].observation
# step 1 = first action step
print(f"Obs Keys {obs.keys()}")
planets = [Planet(*p) for p in obs.planets]
print(f"Player: {obs.player}")
print(f"Angular velocity: {obs.angular_velocity:.4f} rad/turn")
print(f"\nPlanets ({len(planets)}):")
for p in planets[:6]:
    owner_str = f"Player {p.owner}" if p.owner >= 0 else "Neutral"
    print(f"  id={p.id} owner={owner_str:10s} pos=({p.x:.1f}, {p.y:.1f}) r={p.radius:.1f} ships={p.ships} prod={p.production}")

Obs Keys dict_keys(['remainingOverageTime', 'step', 'planets', 'fleets', 'player', 'angular_velocity', 'initial_planets', 'next_fleet_id', 'comets', 'comet_planet_ids'])
Player: 0
Angular velocity: 0.0314 rad/turn

Planets (32):
  id=0 owner=Neutral    pos=(94.9, 75.1) r=2.4 ships=29 prod=4
  id=1 owner=Neutral    pos=(24.9, 94.9) r=2.4 ships=29 prod=4
  id=2 owner=Neutral    pos=(75.1, 5.1) r=2.4 ships=29 prod=4
  id=3 owner=Neutral    pos=(5.1, 24.9) r=2.4 ships=29 prod=4
  id=4 owner=Neutral    pos=(60.7, 98.8) r=1.0 ships=42 prod=1
  id=5 owner=Neutral    pos=(1.2, 60.7) r=1.0 ships=42 prod=1


**Barbell Strategy Plan**



target for today (5/6/26): finalize logic algorithm for barbell strategy + modify helper functions. be specific :)

In [ ]:
import sys
for name in ['compute_distance_rotational', 'compute_distance_static']:
    if name in globals(): del globals()[name]
    if name in locals(): del locals()[name]

In [ ]:
%%writefile submission.py
import math
from typing import List, NamedTuple, Optional, Tuple


class Planet:
    """
    Explicit reconstruction of the Kaggle Environment Planet dataclass 
    to ensure explicit property referencing and structural readability.
    """
    def __init__(self, x: float, y: float, owner: int, ships: int, angular_velocity: float, planet_id: int):
        self.x = x
        self.y = y
        self.owner = owner
        self.ships = ships
        self.angular_velocity = angular_velocity
        self.id = planet_id


class TargetTrajectory(NamedTuple):
    """
    Immutable data architecture capturing calculated intercept coordinates,
    flight profiles, and distances. Replaces fragile, error-prone raw lists.
    """
    origin_planet: Planet
    destination_planet: Planet
    intercept_x: float
    intercept_y: float
    intercept_angle: float
    flight_distance: float
    arrival_time_steps: float


class GlobalResourceBroker:
    """
    Manages centralized ship allocations to strictly enforce the Barbell Strategy.
    Prevents parallel target loops from bankrupting a single home planet's reserves.
    """
    def __init__(self, controlled_planets: List[Planet]):
        self._available_garrisons = {planet.id: planet.ships for planet in controlled_planets}

    def request_allocation(self, origin_id: int, required_ships: int, safety_floor: int) -> int:
        """
        Evaluates and reserves ship counts atomically.
        Returns the exact number of ships authorized for deployment.
        """
        current_balance = self._available_garrisons.get(origin_id, 0)
        spendable_surplus = current_balance - safety_floor

        if spendable_surplus <= 0 or required_ships <= 0:
            return 0

        allocated_amount = min(required_ships, spendable_surplus)
        self._available_garrisons[origin_id] -= allocated_amount
        return allocated_amount


def calculate_orbital_intercept(
    origin: Planet, 
    target: Planet, 
    fleet_speed: float, 
    map_center_x: float, 
    map_center_y: float,
    convergence_tolerance: float = 0.005,
    max_iterations: int = 100
) -> Optional[TargetTrajectory]:
    """
    Executes an iterative Time-Convergent Root-Finding algorithm to predict precise 
    intercept vectors for bodies moving along complex orbital trajectories.
    """
    if fleet_speed <= 0.0:
        return None

    # Step 1: Compute absolute radial properties relative to the designated system focus
    delta_center_x = target.x - map_center_x
    delta_center_y = target.y - map_center_y
    orbital_radius = math.hypot(delta_center_x, delta_center_y)
    initial_orbital_angle = math.atan2(delta_center_y, delta_center_x)

    estimated_flight_time = 0.0
    iteration_count = 0

    # Step 2: Converge flight arrival timeline with target position vectors
    while iteration_count < max_iterations:
        projected_angle = initial_orbital_angle + (target.angular_velocity * estimated_flight_time)
        
        projected_x = map_center_x + (orbital_radius * math.cos(projected_angle))
        projected_y = map_center_y + (orbital_radius * math.sin(projected_angle))

        distance_to_projection = math.hypot(projected_x - origin.x, projected_y - origin.y)
        calculated_flight_time = distance_to_projection / fleet_speed

        # Check for convergence stability
        if abs(calculated_flight_time - estimated_flight_time) < convergence_tolerance:
            return TargetTrajectory(
                origin_planet=origin,
                destination_planet=target,
                intercept_x=projected_x,
                intercept_y=projected_y,
                intercept_angle=projected_angle,
                flight_distance=distance_to_projection,
                arrival_time_steps=calculated_flight_time
            )

        estimated_flight_time = calculated_flight_time
        iteration_count += 1

    return None


def select_optimal_trajectory(
    origins: List[Planet], 
    targets: List[Planet], 
    fleet_speed: float,
    map_center_x: float,
    map_center_y: float
) -> Optional[TargetTrajectory]:
    """
    Scans matrix of permutations to resolve the most mechanically efficient flight path.
    """
    best_trajectory: Optional[TargetTrajectory] = None

    for origin in origins:
        for target in targets:
            trajectory = calculate_orbital_intercept(
                origin, target, fleet_speed, map_center_x, map_center_y
            )
            if not trajectory:
                continue

            if best_trajectory is None or trajectory.flight_distance < best_trajectory.flight_distance:
                best_trajectory = trajectory

    return best_trajectory


def generate_action_vector(origin_id: int, target_x: float, target_y: float, origin_x: float, origin_y: float, ships: int) -> List:
    """
    Generates fully legal environment action coordinates. Corrects previous vector inversion 
    bugs by steering outward explicitly along the origin -> target destination pathway.
    """
    delta_x = target_x - origin_x
    delta_y = target_y - origin_y
    
    # Absolute fallbacks for complete overlap conditions to prevent NaN vector generation
    if math.isclose(delta_x, 0.0) and math.isclose(delta_y, 0.0):
        return [origin_id, 1.0, 0.0, ships]
        
    magnitude = math.hypot(delta_x, delta_y)
    normalized_direction_x = delta_x / magnitude
    normalized_direction_y = delta_y / magnitude

    return [origin_id, normalized_direction_x, normalized_direction_y, ships]


def agent(obs, configuration) -> List:
    """
    Executive execution agent implementing the algorithmic Barbell Strategy.
    Balances low-risk static assertions against aggressive high-variance rotational captures.
    """
    actions = []
    player_id = obs.player
    fleet_speed = getattr(configuration, "maxSpeed", 1.0)
    
    # Defensive Extraction of Map Boundary Characteristics
    map_center_x = getattr(configuration, "mapCenterX", 50.0)
    map_center_y = getattr(configuration, "mapCenterY", 50.0)
    barbell_boundary_radius = getattr(configuration, "innerCoreRadius", 50.0)

    # Hydrate unstructured raw lists into safe, strongly bounded internal classes
    all_planets = [
        Planet(p[0], p[1], p[2], p[3], p[4], idx) 
        for idx, p in enumerate(obs.planets)
    ]

    my_planets = [p for p in all_planets if p.owner == player_id]
    hostile_planets = [p for p in all_planets if p.owner != player_id and p.owner != 0]
    neutral_planets = [p for p in all_planets if p.owner == 0]
    viable_targets = hostile_planets + neutral_planets

    if not my_planets or not viable_targets:
        return []

    # Initialize defensive, centralized inventory bookkeeping
    resource_broker = GlobalResourceBroker(my_planets)

    # Operational Weight 1: Low-Risk Static / Core Targets (Inside Core Radius Boundary)
    static_targets = [
        p for p in viable_targets 
        if math.hypot(p.x - map_center_x, p.y - map_center_y) < barbell_boundary_radius
    ]
    
    # Operational Weight 2: High-Risk Outlier / Rotational Vectors (On Outer System Perimeter)
    rotational_targets = [
        p for p in viable_targets 
        if math.hypot(p.x - map_center_x, p.y - map_center_y) >= barbell_boundary_radius
    ]

    # --- Processing Segment A: Low-Risk / High-Certainty Holds ---
    best_static = select_optimal_trajectory(
        my_planets, static_targets, fleet_speed, map_center_x, map_center_y
    )
    if best_static:
        origin = best_static.origin_planet
        target = best_static.destination_planet
        
        # Low risk: Enforce a strict conservative safety floor to guarantee home defense
        safety_floor = max(30, int(origin.ships * 0.50))
        required_ships = target.ships + 1

        allocated_ships = resource_broker.request_allocation(origin.id, required_ships, safety_floor)
        if allocated_ships > 0:
            actions.append(
                generate_action_vector(
                    origin.id, best_static.intercept_x, best_static.intercept_y, 
                    origin.x, origin.y, allocated_ships
                )
            )

    # --- Processing Segment B: High-Risk / Aggressive Speculations ---
    best_rotational = select_optimal_trajectory(
        my_planets, rotational_targets, fleet_speed, map_center_x, map_center_y
    )
    if best_rotational:
        origin = best_rotational.origin_planet
        target = best_rotational.destination_planet

        # High risk execution: Lower home defense floor to free up massive speculative assets
        safety_floor = max(12, int(origin.ships * 0.15))
        
        # Compensate for long flight times by heavily over-allocating force to ensure capture
        required_ships = int((target.ships + 1) * 2.2)

        allocated_ships = resource_broker.request_allocation(origin.id, required_ships, safety_floor)
        if allocated_ships > 0:
            actions.append(
                generate_action_vector(
                    origin.id, best_rotational.intercept_x, best_rotational.intercept_y, 
                    origin.x, origin.y, allocated_ships
                )
            )

    return actions